In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

In [4]:

# load files
sales = pd.read_csv("data/sales_train_validation.csv")
calendar = pd.read_csv("data/calendar.csv")
sell_prices = pd.read_csv("data/sell_prices.csv")

# keep id columns separately
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

# wide -> long
sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# join calendar info
df = sales_long.merge(calendar, on="d", how="left")

# join prices using Walmart week key
df = df.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# date handling
df["date"] = pd.to_datetime(df["date"])

# reduce memory a bit
for col in ["sales", "sell_price"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

# keep categorical columns as category dtype
cat_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

# sort before feature engineering
df = df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)


In [5]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price'],
      dtype='object')

In [8]:
print(df.isnull().sum())

id                     0
item_id                0
dept_id                0
cat_id                 0
store_id               0
state_id               0
d                      0
sales                  0
date                   0
wm_yr_wk               0
weekday                0
wday                   0
month                  0
year                   0
event_name_1    53631910
event_type_1    53631910
event_name_2    58205410
event_type_2    58205410
snap_CA                0
snap_TX                0
snap_WI                0
sell_price      12299413
dtype: int64


In [14]:
df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,is_weekend,dayofweek,is_high_impact_event
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3.0,2011-01-29,11101,...,NaN,NaN,NaN,0,0,0,2.0,1,6,0
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0.0,2011-01-30,11101,...,NaN,NaN,NaN,0,0,0,2.0,1,7,0
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0.0,2011-01-31,11101,...,NaN,NaN,NaN,0,0,0,2.0,0,1,0
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1.0,2011-02-01,11101,...,NaN,NaN,NaN,1,1,0,2.0,0,2,0
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4.0,2011-02-02,11101,...,NaN,NaN,NaN,1,0,1,2.0,0,3,0


## Feature Engineering

### Calendar Features

In [11]:
# 1. is_weekend
df['is_weekend'] = df['weekday'].isin(
    ['Friday', 'Saturday', 'Sunday']
).astype(int)

# 2. dayofweek（Map them to numerical value）
day_map = {
    'Monday': 1, 'Tuesday': 2, 'Wednesday': 3,
    'Thursday': 4, 'Friday': 5, 'Saturday': 6, 'Sunday': 7
}
df['dayofweek'] = df['weekday'].map(day_map)

# 3. is_high_impact_event
high_impact_events = ['LaborDay', 'SuperBowl', 'Easter']
df['is_high_impact_event'] = df['event_name_1'].isin(
    high_impact_events
).astype(int)


In [12]:
# Check
print(df[['weekday', 'is_weekend', 'dayofweek', 'is_high_impact_event']].head(10))

     weekday  is_weekend  dayofweek  is_high_impact_event
0   Saturday           1          6                     0
1     Sunday           1          7                     0
2     Monday           0          1                     0
3    Tuesday           0          2                     0
4  Wednesday           0          3                     0
5   Thursday           0          4                     0
6     Friday           1          5                     0
7   Saturday           1          6                     0
8     Sunday           1          7                     1
9     Monday           0          1                     0


### Price Features

In [17]:
# 1. fill NaN first
df['sell_price'] = df.groupby(
    ['store_id', 'item_id']
)['sell_price'].transform(lambda x: x.ffill())

# 2. price_vs_mean
df['price_vs_mean'] = df.groupby(
    'item_id'
)['sell_price'].transform(lambda x: x / x.mean())

# 3. price_change
df['price_change'] = df.groupby(
    ['store_id', 'item_id']
)['sell_price'].transform(lambda x: x.diff())

# 4. price_momentum
df['price_momentum'] = df.groupby(
    ['store_id', 'item_id']
)['price_change'].transform(
    lambda x: x.rolling(28).sum()
)


/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/1286831569.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['sell_price'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/1286831569.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['price_vs_mean'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/1286831569.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default an

In [18]:
# Check
print(df[['sell_price', 'price_vs_mean', 'price_change', 'price_momentum']].head(10))

   sell_price  price_vs_mean  price_change  price_momentum
0         2.0       0.924094           NaN             NaN
1         2.0       0.924094           0.0             NaN
2         2.0       0.924094           0.0             NaN
3         2.0       0.924094           0.0             NaN
4         2.0       0.924094           0.0             NaN
5         2.0       0.924094           0.0             NaN
6         2.0       0.924094           0.0             NaN
7         2.0       0.924094           0.0             NaN
8         2.0       0.924094           0.0             NaN
9         2.0       0.924094           0.0             NaN


In [19]:
# Check more rows
print(df['price_momentum'].first_valid_index())
print(df[['sell_price', 'price_change', 'price_momentum']].iloc[20:35])

28
    sell_price  price_change  price_momentum
20         2.0           0.0             NaN
21         2.0           0.0             NaN
22         2.0           0.0             NaN
23         2.0           0.0             NaN
24         2.0           0.0             NaN
25         2.0           0.0             NaN
26         2.0           0.0             NaN
27         2.0           0.0             NaN
28         2.0           0.0             0.0
29         2.0           0.0             0.0
30         2.0           0.0             0.0
31         2.0           0.0             0.0
32         2.0           0.0             0.0
33         2.0           0.0             0.0
34         2.0           0.0             0.0


### Lag Features

In [20]:
lag_days = [7, 14, 28, 35, 42, 364, 365]

for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby(
        ['store_id', 'item_id']
    )['sales'].transform(lambda x: x.shift(lag))


/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/298651144.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'lag_{lag}'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/298651144.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'lag_{lag}'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/298651144.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silen

In [21]:
# Check
print(df[['sales', 'lag_7', 'lag_28', 'lag_365']].head(20))

    sales  lag_7  lag_28  lag_365
0     3.0    NaN     NaN      NaN
1     0.0    NaN     NaN      NaN
2     0.0    NaN     NaN      NaN
3     1.0    NaN     NaN      NaN
4     4.0    NaN     NaN      NaN
5     2.0    NaN     NaN      NaN
6     0.0    NaN     NaN      NaN
7     2.0    3.0     NaN      NaN
8     0.0    0.0     NaN      NaN
9     0.0    0.0     NaN      NaN
10    0.0    1.0     NaN      NaN
11    0.0    4.0     NaN      NaN
12    3.0    2.0     NaN      NaN
13    1.0    0.0     NaN      NaN
14    3.0    2.0     NaN      NaN
15    0.0    0.0     NaN      NaN
16    2.0    0.0     NaN      NaN
17    1.0    0.0     NaN      NaN
18    2.0    0.0     NaN      NaN
19    0.0    3.0     NaN      NaN


### Rolling Features

In [22]:
# rolling mean
df['rolling_mean_7'] = df.groupby(
    ['store_id', 'item_id']
)['sales'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df['rolling_mean_28'] = df.groupby(
    ['store_id', 'item_id']
)['sales'].transform(
    lambda x: x.shift(1).rolling(28).mean()
)

df['rolling_std_7'] = df.groupby(
    ['store_id', 'item_id']
)['sales'].transform(
    lambda x: x.shift(1).rolling(7).std()
)


/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/3353393817.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['rolling_mean_7'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/3353393817.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['rolling_mean_28'] = df.groupby(
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_34055/3353393817.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future defa

In [23]:
# Check
print(df[['sales', 'rolling_mean_7', 'rolling_mean_28']].head(40))

    sales  rolling_mean_7  rolling_mean_28
0     3.0             NaN              NaN
1     0.0             NaN              NaN
2     0.0             NaN              NaN
3     1.0             NaN              NaN
4     4.0             NaN              NaN
5     2.0             NaN              NaN
6     0.0             NaN              NaN
7     2.0        1.428571              NaN
8     0.0        1.285714              NaN
9     0.0        1.285714              NaN
10    0.0        1.285714              NaN
11    0.0        1.142857              NaN
12    3.0        0.571429              NaN
13    1.0        0.714286              NaN
14    3.0        0.857143              NaN
15    0.0        1.000000              NaN
16    2.0        1.000000              NaN
17    1.0        1.285714              NaN
18    2.0        1.428571              NaN
19    0.0        1.714286              NaN
20    2.0        1.285714              NaN
21    1.0        1.428571              NaN
22    2.0  

In [24]:
print(df.shape)
print(df.columns.tolist())

(58327370, 38)
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'is_weekend', 'dayofweek', 'is_high_impact_event', 'price_vs_mean', 'price_change', 'price_momentum', 'lag_7', 'lag_14', 'lag_28', 'lag_35', 'lag_42', 'lag_364', 'lag_365', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7']
